# Credit Default Prediction Model

**Objective:** Predict probability of loan default within 18 months using external bureau features (Experian Premier Attributes, ClearView, vendor scores).

**Workflow:**
1. Data Exploration & Validation
2. Model Training (XGBoost)
3. Evaluation (AUC, KS, Feature Importance)
4. Experiment Tracking
5. Model Registry & SQL Inference

## 1. Data Exploration

In [ ]:
%%sql -r data_overview
SELECT COUNT(*) AS total_rows,
       ROUND(AVG(DEFAULT_18M), 3) AS default_rate,
       ROUND(AVG(FICO_SCORE), 1) AS avg_fico,
       ROUND(AVG(REVOLVING_UTILIZATION_PCT), 1) AS avg_utilization,
       ROUND(AVG(CASE WHEN NUM_DELINQ_30_12M = 0 THEN 1 ELSE 0 END), 3) AS pct_zero_delinq
FROM CREDIT_RISK.ML.LOAN_APPLICATIONS

In [ ]:
%%sql -r default_by_fico
SELECT 
    CASE 
        WHEN FICO_SCORE < 580 THEN '1: <580 (Deep Subprime)'
        WHEN FICO_SCORE < 620 THEN '2: 580-619 (Subprime)'
        WHEN FICO_SCORE < 670 THEN '3: 620-669 (Near Prime)'
        WHEN FICO_SCORE < 740 THEN '4: 670-739 (Prime)'
        WHEN FICO_SCORE < 800 THEN '5: 740-799 (Super Prime)'
        ELSE '6: 800+ (Exceptional)'
    END AS FICO_BAND,
    COUNT(*) AS N,
    ROUND(AVG(DEFAULT_18M), 3) AS DEFAULT_RATE
FROM CREDIT_RISK.ML.LOAN_APPLICATIONS
GROUP BY 1
ORDER BY 1

In [ ]:
%%sql -r correlations
SELECT 
    ROUND(CORR(FICO_SCORE, VANTAGE_SCORE), 3) AS FICO_VANTAGE,
    ROUND(CORR(FICO_SCORE, REVOLVING_UTILIZATION_PCT), 3) AS FICO_UTIL,
    ROUND(CORR(FICO_SCORE, NUM_DELINQ_30_12M), 3) AS FICO_DELINQ,
    ROUND(CORR(REVOLVING_UTILIZATION_PCT, DEFAULT_18M), 3) AS UTIL_DEFAULT,
    ROUND(CORR(FICO_SCORE, DEFAULT_18M), 3) AS FICO_DEFAULT
FROM CREDIT_RISK.ML.LOAN_APPLICATIONS

## 2. Model Training

Train an XGBoost classifier on 70% of the data, evaluate on 30% holdout.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from xgboost import XGBClassifier

session = get_active_session()
session.use_database("CREDIT_RISK")
session.use_schema("ML")

# Load data
df = session.table("LOAN_APPLICATIONS").to_pandas()

# Define features and target
feature_cols = [
    'FICO_SCORE', 'VANTAGE_SCORE', 'NUM_OPEN_TRADES', 'NUM_TRADES_EVER',
    'REVOLVING_UTILIZATION_PCT', 'TOTAL_REVOLVING_BALANCE',
    'NUM_DELINQ_30_12M', 'NUM_DELINQ_60_12M', 'NUM_DELINQ_90_EVER',
    'MONTHS_SINCE_OLDEST_TRADE', 'MONTHS_SINCE_NEWEST_TRADE', 'NUM_INQUIRIES_6M',
    'BANKRUPTCY_FLAG', 'TOTAL_INSTALLMENT_BALANCE',
    'PAYMENT_VELOCITY_TREND', 'BALANCE_TREND_12M', 'VENDOR_BANKRUPTCY_SCORE'
]

X = df[feature_cols]
y = df['DEFAULT_18M']

# Train/test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print(f"Default rate (train): {y_train.mean():.3f}")
print(f"Default rate (test): {y_test.mean():.3f}")

In [ ]:
# Train XGBoost
params = {
    'n_estimators': 200,
    'max_depth': 5,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'auc',
    'random_state': 42,
    'use_label_encoder': False
}

model = XGBClassifier(**params)
model.fit(X_train, y_train)

# Predict probabilities on test set
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("Model trained successfully.")

## 3. Model Evaluation

In [ ]:
# Calculate key metrics
auc_score = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
ks_stat = max(tpr - fpr)
gini = 2 * auc_score - 1

print("=" * 40)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 40)
print(f"AUC:            {auc_score:.4f}")
print(f"KS Statistic:   {ks_stat:.4f}")
print(f"Gini:           {gini:.4f}")
print("=" * 40)
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

In [ ]:
import matplotlib.pyplot as plt

# Feature importance
importance = model.feature_importances_
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': importance})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='steelblue')
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('XGBoost Feature Importance - Credit Default Model')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'XGBoost (AUC = {auc_score:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - Credit Default Model')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Experiment Tracking

Log the training run to Snowflake's experiment tracking framework for reproducibility and comparison.

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(
    session=session,
    database_name="CREDIT_RISK",
    schema_name="ML"
)
exp.set_experiment("CREDIT_DEFAULT_EXPERIMENT")

with exp.start_run("xgboost_v1"):
    # Log hyperparameters
    exp.log_params(params)
    exp.log_params({
        'train_rows': X_train.shape[0],
        'test_rows': X_test.shape[0],
        'num_features': len(feature_cols),
        'target_default_rate': round(float(y_train.mean()), 3)
    })
    
    # Log evaluation metrics
    exp.log_metrics({
        'auc': round(auc_score, 4),
        'ks_statistic': round(ks_stat, 4),
        'gini': round(gini, 4)
    })

print("Experiment run logged successfully.")

## 5. Register Model to Snowflake Model Registry

Register the trained model so it can be called via SQL `!PREDICT()` — no containers, no endpoints, just SQL.

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="CREDIT_RISK", schema_name="ML")

# Register model with sample input for schema inference
mv = reg.log_model(
    model,
    model_name="CREDIT_DEFAULT_MODEL",
    version_name="v1",
    sample_input_data=X_test.head(10),
    conda_dependencies=["xgboost", "scikit-learn"],
    target_platforms=["WAREHOUSE"],
    comment=f"XGBoost credit default model - bureau features only. AUC={auc_score:.4f}, KS={ks_stat:.4f}"
)

print(f"Model registered: {mv.model_name} version {mv.version_name}")
print(f"Functions available: {mv.show_functions()}")

## 6. SQL Inference

Score new applicants directly with SQL — this is how production would use the model.

In [ ]:
%%sql -r sql_inference
SELECT 
    APPLICANT_ID,
    FICO_SCORE,
    REVOLVING_UTILIZATION_PCT,
    NUM_DELINQ_30_12M,
    CREDIT_RISK.ML.CREDIT_DEFAULT_MODEL!PREDICT_PROBA(
        FICO_SCORE, VANTAGE_SCORE, NUM_OPEN_TRADES, NUM_TRADES_EVER,
        REVOLVING_UTILIZATION_PCT, TOTAL_REVOLVING_BALANCE,
        NUM_DELINQ_30_12M, NUM_DELINQ_60_12M, NUM_DELINQ_90_EVER,
        MONTHS_SINCE_OLDEST_TRADE, MONTHS_SINCE_NEWEST_TRADE, NUM_INQUIRIES_6M,
        BANKRUPTCY_FLAG, TOTAL_INSTALLMENT_BALANCE,
        PAYMENT_VELOCITY_TREND, BALANCE_TREND_12M, VENDOR_BANKRUPTCY_SCORE
    ):output_feature_1::FLOAT AS PROB_DEFAULT
FROM CREDIT_RISK.ML.LOAN_APPLICATIONS
LIMIT 10

## 7. Approve/Decline Decision Logic

Apply a business rule: decline applicants with predicted default probability > 35%.

In [ ]:
%%sql -r credit_decisions
WITH scored AS (
    SELECT 
        APPLICANT_ID,
        FICO_SCORE,
        CREDIT_RISK.ML.CREDIT_DEFAULT_MODEL!PREDICT_PROBA(
            FICO_SCORE, VANTAGE_SCORE, NUM_OPEN_TRADES, NUM_TRADES_EVER,
            REVOLVING_UTILIZATION_PCT, TOTAL_REVOLVING_BALANCE,
            NUM_DELINQ_30_12M, NUM_DELINQ_60_12M, NUM_DELINQ_90_EVER,
            MONTHS_SINCE_OLDEST_TRADE, MONTHS_SINCE_NEWEST_TRADE, NUM_INQUIRIES_6M,
            BANKRUPTCY_FLAG, TOTAL_INSTALLMENT_BALANCE,
            PAYMENT_VELOCITY_TREND, BALANCE_TREND_12M, VENDOR_BANKRUPTCY_SCORE
        ):output_feature_1::FLOAT AS PROB_DEFAULT,
        DEFAULT_18M AS ACTUAL_DEFAULT
    FROM CREDIT_RISK.ML.LOAN_APPLICATIONS
)
SELECT 
    APPLICANT_ID,
    FICO_SCORE,
    ROUND(PROB_DEFAULT, 3) AS PROB_DEFAULT,
    CASE WHEN PROB_DEFAULT > 0.35 THEN 'DECLINE' ELSE 'APPROVE' END AS DECISION,
    CASE WHEN ACTUAL_DEFAULT = 1 THEN 'Defaulted' ELSE 'No Default' END AS ACTUAL_OUTCOME
FROM scored
LIMIT 20